In [4]:
def clean_tokenize(text):
    text = str(text).lower()
    text = text.replace('\\n', ' ')                     # literal backslash-n from JSON export
    text = re.sub(r'http\S+|www\.\S+', '', text)         # URLs
    text = re.sub(r'@\w+', '', text)                     # mentions
    text = re.sub(r'#\w+', '', text)                     # hashtags (handled separately)
    text = re.sub(r'[^a-z\s]', ' ', text)                 # strip punctuation/numbers/emoji
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

df['tokens'] = df['full_text'].apply(clean_tokenize)

In [5]:
df[['full_text', 'tokens']].sample(10, random_state=1)

,full_text,tokens
120,I awe to this resistance fighter in West Bank ...,"[awe, resistance, fighter, west, bank, attacke..."
374,A Palestinian child confronts Israeli settlers...,"[palestinian, child, confronts, israeli, settl..."
90,Palestinians placed rocks to block a road in t...,"[palestinians, placed, rocks, block, road, occ..."
844,"An attack on healthcare happens, on average, e...","[attack, healthcare, happens, average, every, ..."
649,𝟖𝟔-𝐒𝐄𝐀𝐓 𝐄𝐔𝐑𝐎𝐏𝐄𝐀𝐍 𝐁𝐋𝐎𝐂 𝐁𝐀𝐂𝐊𝐒 𝐈𝐒𝐑𝐀𝐄𝐋 𝐈𝐍 𝐉𝐔𝐃𝐄𝐀 𝐀𝐍...,"[seat, bloc, european, parliament, told, israe..."
311,It's so simple 👇\n\nPalestinians are NOT victi...,"[simple, palestinians, victims, way, ukrainian..."
180,Israel continues acting as a rogue genocidial ...,"[israel, continues, acting, rogue, genocidial,..."
119,Amazing bit of propaganda engineering here.\n\...,"[amazing, bit, propaganda, engineering, even, ..."
49,Kristen Recommends!\nIf you're looking for you...,"[kristen, recommends, looking, next, captivati..."
101,#Israel tightens #Jerusalem and #WestBank rest...,"[tightens, restrictions, israeli, authorities,..."


In [6]:
from collections import Counter
from itertools import chain

# Overall frequency
all_tokens = list(chain.from_iterable(df['tokens']))
overall_freq = Counter(all_tokens)

print(overall_freq.most_common(30))

[('west', 1242), ('bank', 1221), ('israel', 955), ('israeli', 712), ('gaza', 650), ('palestinian', 580), ('people', 374), ('palestinians', 305), ('war', 276), ('one', 276), ('state', 275), ('jewish', 265), ('palestine', 265), ('said', 254), ('occupied', 242), ('world', 232), ('hamas', 227), ('also', 222), ('would', 207), ('god', 194), ('iran', 187), ('sheeran', 177), ('genocide', 175), ('military', 172), ('two', 172), ('international', 169), ('forces', 169), ('killed', 166), ('attack', 164), ('children', 163)]


In [7]:
# Per-country frequency
country_freq = {}
for country, group in df.groupby('country'):
    tokens = list(chain.from_iterable(group['tokens']))
    country_freq[country] = Counter(tokens)

# check post-count per country first — this tells us which countries
# even have enough volume to look at meaningfully
df['country'].value_counts()

country
United States          415
United Kingdom         148
Canada                  61
Australia               47
Israel                  38
                      ... 
Trinidad and Tobago      1
Ghana                    1
Luxembourg               1
Vietnam                  1
Zambia                   1
Name: count, Length: 76, dtype: int64

In [10]:
MIN_POSTS = 20
eligible_countries = df['country'].value_counts()
eligible_countries = eligible_countries[eligible_countries >= MIN_POSTS].index.tolist()

top_terms_by_country = {}
for country in eligible_countries:
    tokens = list(chain.from_iterable(df[df['country'] == country]['tokens']))
    top_terms_by_country[country] = Counter(tokens).most_common(20)

for country, terms in top_terms_by_country.items():
    print(country, '->', terms[:10])

United States -> [('israel', 492), ('west', 485), ('bank', 477), ('gaza', 275), ('palestinian', 232), ('israeli', 214), ('people', 213), ('god', 187), ('jewish', 169), ('war', 155)]
United Kingdom -> [('west', 172), ('bank', 171), ('israeli', 122), ('israel', 120), ('gaza', 99), ('palestinian', 66), ('palestine', 44), ('occupied', 40), ('palestinians', 39), ('forces', 37)]
Canada -> [('west', 69), ('bank', 66), ('israel', 52), ('israeli', 40), ('gaza', 31), ('children', 26), ('palestinian', 20), ('said', 20), ('canada', 18), ('sheeran', 16)]
Australia -> [('west', 54), ('bank', 52), ('israel', 43), ('gaza', 30), ('israeli', 25), ('palestinian', 16), ('even', 13), ('people', 13), ('one', 13), ('state', 12)]
Israel -> [('bank', 48), ('west', 47), ('palestinian', 31), ('israel', 27), ('land', 26), ('israeli', 20), ('attack', 18), ('area', 14), ('state', 14), ('palestinians', 13)]
India -> [('west', 43), ('bank', 41), ('india', 27), ('israeli', 27), ('gaza', 23), ('israel', 22), ('palestin

In [11]:
top_terms_by_country_normalized = {}
for country in eligible_countries:
    tokens = list(chain.from_iterable(df[df['country'] == country]['tokens']))
    total = len(tokens)
    counts = Counter(tokens).most_common(20)
    top_terms_by_country_normalized[country] = [(word, count, round(count/total*100, 2)) for word, count in counts]

for country, terms in top_terms_by_country_normalized.items():
    print(country, '->', terms[:5])

United States -> [('israel', 492, 1.26), ('west', 485, 1.24), ('bank', 477, 1.22), ('gaza', 275, 0.7), ('palestinian', 232, 0.59)]
United Kingdom -> [('west', 172, 2.25), ('bank', 171, 2.24), ('israeli', 122, 1.59), ('israel', 120, 1.57), ('gaza', 99, 1.29)]
Canada -> [('west', 69, 2.33), ('bank', 66, 2.23), ('israel', 52, 1.76), ('israeli', 40, 1.35), ('gaza', 31, 1.05)]
Australia -> [('west', 54, 2.33), ('bank', 52, 2.24), ('israel', 43, 1.86), ('gaza', 30, 1.29), ('israeli', 25, 1.08)]
Israel -> [('bank', 48, 2.4), ('west', 47, 2.35), ('palestinian', 31, 1.55), ('israel', 27, 1.35), ('land', 26, 1.3)]
India -> [('west', 43, 1.56), ('bank', 41, 1.49), ('india', 27, 0.98), ('israeli', 27, 0.98), ('gaza', 23, 0.83)]
Philippines -> [('west', 23, 3.42), ('bank', 22, 3.27), ('israeli', 15, 2.23), ('palestinian', 13, 1.93), ('actions', 11, 1.63)]
Ireland -> [('west', 21, 4.1), ('bank', 20, 3.91), ('gaza', 10, 1.95), ('israeli', 8, 1.56), ('israel', 7, 1.37)]


In [12]:
import json

# Overall frequency
with open('freq_overall.json', 'w') as f:
    json.dump(overall_freq.most_common(50), f, indent=2)

# Per-country (raw + normalized), only eligible countries (≥20 posts)
freq_by_country_export = {
    country: top_terms_by_country_normalized[country]
    for country in eligible_countries
}
with open('freq_by_country.json', 'w') as f:
    json.dump(freq_by_country_export, f, indent=2)

print("Saved freq_overall.json and freq_by_country.json")

Saved freq_overall.json and freq_by_country.json
